# Iterative Temporal Parent Discovery

Dev | Jacqueline Maasch | August 2026

In [1]:
# General imports.
import pandas as pd
import numpy as np
#import pickle
import matplotlib.pyplot as plt
import time
from string import ascii_uppercase as alphabet

# Time series causal discovery.
import tigramite
from tigramite import data_processing as pp
from tigramite.toymodels import structural_causal_processes as toys
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI # also for PCMCI+
from tigramite.independence_tests.gsquared import Gsquared # univariate discrete/categorical vars
from tigramite.independence_tests.cmisymb import CMIsymb # multivariate discrete/categorical vars (permutation-based)
from tigramite.independence_tests.parcorr import ParCorr # univariate, continuous variables with linear dependencies and Gaussian noise

# VLCD.
from vlcd import PCMCIConfig, pcmci

# Vanilla causal discovery.
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.cit import CIT

# Custom discovery algorithms.
from padl import PaDL
from itpd_naive import ITPDNaive, Utils

# Metrics.
#from dodiscover.metrics import structure_hamming_dist as SHD
from cdt.metrics import precision_recall
from cdt.metrics import SHD
from cdt.metrics import SID

# Graphical modeling.
import networkx as nx

No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


# ITPD

## Generate random time series

In [2]:
# Get random data.
u = Utils()
dfs = []
graphs = []
adj_matrices = []
p2c_list = []
c2p_list = []
M = 1_000
T = 7
N = 10
max_children = 2
plot = False
iters = 10 # total random datasets to generate.

for i in range(iters):
    df, p2c, c2p, G, true_adj = u.get_data(M = M, 
                                           T = T,
                                           N = N, 
                                           max_children = max_children,
                                           plot = plot)
    dfs.append(df)
    graphs.append(G)
    adj_matrices.append(true_adj)
    p2c_list.append(p2c)
    c2p_list.append(c2p)

In [3]:
dfs[0]

,A0,A1,A2,A3,A4,A5,A6,B0,B1,B2,...,I4,I5,I6,J0,J1,J2,J3,J4,J5,J6
0,-0.847999,1.029932,-1.193852,1.315783,-1.373237,5.920172,-2.639018,0.974093,-3.069736,-5.066763,...,-0.945008,0.186254,4.788573,-1.174345,2.637315,0.495725,0.374204,0.769240,1.603390,-1.820623
1,-0.452096,-1.473262,0.561195,-2.809776,4.200006,-4.580619,3.026212,-2.258242,3.218657,3.645229,...,-0.306525,1.620473,-3.223719,-0.483129,2.000129,-1.274755,-0.133903,3.988172,5.939352,-8.393322
2,0.011015,1.384411,-2.784778,3.039437,-4.031397,9.119231,-9.863248,-0.236077,0.370553,0.504988,...,15.267414,-22.775968,-13.193911,-0.523622,0.497184,0.875926,0.604611,-4.215684,-8.275877,12.561723
3,1.171663,-1.899749,1.699085,-1.845876,4.107431,-7.201105,3.639085,-0.932244,1.421501,3.484536,...,1.656770,-1.804717,-11.920464,0.879847,-0.829733,-1.462498,1.486202,1.204584,3.097990,-6.377161
4,-0.378504,0.493159,-0.510259,1.565841,-0.011122,4.671782,-2.463687,0.802775,-0.370323,0.769454,...,-5.138845,8.291578,12.345335,1.367032,-0.913321,-3.854947,5.473199,-5.159667,-12.951861,20.154851
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.387148,0.589968,-0.686239,0.539577,-1.243371,3.506657,-7.543375,0.002519,0.904806,-1.025932,...,5.133785,-9.816126,-8.135306,-0.239460,1.367201,1.851397,-3.859235,-3.643956,-6.083071,8.346517
996,1.048701,-1.259609,1.462401,-2.155168,1.686683,-5.499259,8.992486,-0.499612,0.996955,1.348666,...,-8.170824,8.956838,3.191714,-0.139119,-1.136890,-0.482615,-0.309327,1.933297,3.526337,-3.376478
997,-0.998251,1.944212,-1.710171,3.615194,-6.443426,8.491348,-6.824296,-1.695076,3.919816,6.985589,...,4.169756,-9.088180,-9.214235,-0.546189,0.349348,-2.516201,5.234392,-7.642060,-19.712490,28.391207
998,-1.831408,-1.303955,1.310306,-2.068002,4.160726,0.532968,-3.611357,0.504858,-0.331256,-0.312148,...,2.034113,-3.209831,-9.026208,0.438526,-2.263245,-0.433459,1.566228,-4.429006,-6.467134,10.589185


In [4]:
p2c_list[0]

{'A0': ['A1', 'J2'],
 'A1': ['A2', 'G2', 'A4'],
 'A2': ['A3', 'D4', 'F3'],
 'A3': ['A4', 'B5'],
 'A4': ['A5', 'B5', 'F5'],
 'A5': ['A6', 'F6'],
 'A6': [],
 'B0': ['B1'],
 'B1': ['B2', 'C5'],
 'B2': ['B3'],
 'B3': ['B4', 'F4', 'B6'],
 'B4': ['B5', 'A5'],
 'B5': ['B6', 'D6'],
 'B6': [],
 'C0': ['C1', 'I6'],
 'C1': ['C2', 'B2', 'J4'],
 'C2': ['C3', 'A6'],
 'C3': ['C4'],
 'C4': ['C5', 'E5'],
 'C5': ['C6', 'B6'],
 'C6': [],
 'D0': ['D1'],
 'D1': ['D2', 'D4', 'G4'],
 'D2': ['D3', 'I4', 'A6'],
 'D3': ['D4', 'H6'],
 'D4': ['D5', 'C5'],
 'D5': ['D6', 'E6', 'I6'],
 'D6': [],
 'E0': ['E1', 'D1'],
 'E1': ['E2'],
 'E2': ['E3', 'G5'],
 'E3': ['E4', 'J5', 'C6'],
 'E4': ['E5', 'A5', 'A5'],
 'E5': ['E6', 'H6', 'F6'],
 'E6': [],
 'F0': ['F1', 'F6', 'F2'],
 'F1': ['F2'],
 'F2': ['F3', 'E5'],
 'F3': ['F4', 'E5'],
 'F4': ['F5', 'G6', 'B5'],
 'F5': ['F6', 'C6'],
 'F6': [],
 'G0': ['G1', 'D2', 'I5'],
 'G1': ['G2'],
 'G2': ['G3'],
 'G3': ['G4'],
 'G4': ['G5', 'H5'],
 'G5': ['G6', 'E6', 'B6'],
 'G6': [],
 'H0'

## Oracle

In [5]:
#%%capture

#'''
test = "oracle"
alpha = 0.01
tau_max = None
pairs = None
var_names = list(alphabet[:N])

accuracies = []
auprs = []
shds = []
total_tests = []
cond_sizes = []
itpd_oracle_times = []

for i in range(iters):
    start = time.time()
    it = ITPDNaive(dfs[i],
                  var_names = var_names,
                  pairs = pairs,
                  independence_test = test,
                  alpha = alpha,
                  tau_max = tau_max,
                  true_adj = adj_matrices[i])
    accuracy, aupr, shd = it.itpd_naive()
    itpd_oracle_times.append(time.time() - start)
    
    total_tests.append(np.sum(it.total_tests))
    #cond_sizes += it.conditioning_set_sizes
    
    accuracies.append(accuracy)
    auprs.append(aupr)
    shds.append(shd)

print()
print(f"Mean runtime     : {round(np.mean(itpd_oracle_times),4)} seconds")
print(f"Mean total tests : {round(np.mean(total_tests),2)}")
#print(f"Mean cond. set : {round(np.mean(cond_sizes),2)}")
print()
print(f"Mean accuracy    : {round(np.mean(accuracies)*100,2)}")
print(f"Mean AUPR        : {round(np.mean(auprs)*100,2)}")
print(f"Mean SHD         : {round(np.mean(shds),2)}")
#'''

ITPD complete in 0.834 seconds.
ITPD complete in 0.6729 seconds.
ITPD complete in 0.5935 seconds.
ITPD complete in 0.6886 seconds.
ITPD complete in 0.5453 seconds.
ITPD complete in 0.6349 seconds.
ITPD complete in 0.6757 seconds.
ITPD complete in 0.7662 seconds.
ITPD complete in 0.6879 seconds.
ITPD complete in 0.6521 seconds.

Mean runtime     : 0.6761 seconds
Mean total tests : 6587.6

Mean accuracy    : 100.0
Mean AUPR        : 100.0
Mean SHD         : 0.0


## From data

In [6]:
test = "fisherz"
alpha = 0.01
tau_max = None
pairs = None
var_names = list(alphabet[:N])

accuracies = []
auprs = []
shds = []
total_tests = []
cond_sizes = []
itpd_times = []

for i in range(iters):
    start = time.time()
    it = ITPD(dfs[i],
              var_names = var_names,
              pairs = pairs,
              independence_test = test,
              alpha = alpha,
              tau_max = tau_max,
              true_adj = adj_matrices[i])
    accuracy, aupr, shd = it.itpd_naive()
    itpd_times.append(time.time() - start)
    
    total_tests.append(np.sum(it.total_tests))
    #cond_sizes += it.conditioning_set_sizes
    
    accuracies.append(accuracy)
    auprs.append(aupr)
    shds.append(shd)

print()
print(f"Mean runtime     : {round(np.mean(itpd_times),4)} seconds")
print(f"Mean total tests : {round(np.mean(total_tests),2)}")
#print(f"Mean cond. set : {round(np.mean(cond_sizes),2)}")
print()
print(f"Mean accuracy    : {round(np.mean(accuracies)*100,2)}")
print(f"Mean AUPR        : {round(np.mean(auprs)*100,2)}")
print(f"Mean SHD         : {round(np.mean(shds),2)}")

NameError: name 'ITPD' is not defined

In [ ]:
print(f"(N*T)**2 / mean total tests: {(N*T)**2} / {round(np.mean(total_tests),2)} = {round(((N*T)**2) / np.mean(total_tests),2)}")

In [ ]:
plt.hist(auprs)

In [ ]:
plt.hist(accuracies)

# PCMCI

In [ ]:
%%capture
'''
# Sanity check (passed).
m_dict = u.format_tigramite(dfs[0],
                            var_names = var_names)
print(m_dict[0].shape)
print(len(m_dict.keys()))
print(M)
print(T)
print(N)
print(m_dict[1])
dfs[0]
'''

In [ ]:
TAU_MIN = 1
TAU_MAX = 3

pcmci_times = []
for i in range(iters):

    m_dict = u.format_tigramite(dfs[i],
                                var_names = var_names)
    df_tigra = pp.DataFrame(m_dict, 
                            analysis_mode = "multiple", 
                            var_names = var_names)

    pcmci = PCMCI(dataframe = df_tigra, cond_ind_test = ParCorr())
    
    start = time.time()
    results = pcmci.run_pcmci(tau_min = TAU_MIN,
                              tau_max = TAU_MAX, 
                              pc_alpha = 0.01)
    pcmci_runtime = time.time()-start
    pcmci_times.append(pcmci_runtime)
    print(f"PCMCI complete in {round(pcmci_runtime,4)} seconds.")
    
    '''
    tp.plot_graph(graph=results['graph'], 
                  val_matrix=results['val_matrix'], 
                  var_names=var_names)
    plt.show()
    '''

print()
print(f"Mean runtime     : {round(np.mean(pcmci_times),4)} seconds")

In [ ]:
print(f"pcmci mean time / itpd mean time = {round(np.mean(pcmci_times) / np.mean(itpd_times),4)}")

# End of document